In [ ]:
 
import os
import tensorflow_data_validation as tfdv
from tfx.types import artifact_utils
from tfx.types.standard_artifacts import Statistics, Schema, ExampleAnomalies

In [ ]:
from tensorflow_metadata.proto.v0 import statistics_pb2

with open('/root/airflow/dags/COMP315_Group8_project/outputs/StatisticsGen/statistics/2/Split-train/FeatureStats.pb', 'rb') as f:
    raw = f.read()

stats_list = statistics_pb2.DatasetFeatureStatisticsList()
stats_list.ParseFromString(raw)

print(f"Number of datasets: {len(stats_list.datasets)}")

for dataset in stats_list.datasets:
    print(f"\n📊 Dataset: '{dataset.name}' | Rows: {dataset.num_examples}")
    print("=" * 60)
    for i, feature in enumerate(dataset.features):
        fname = feature.name if feature.name else ' > '.join(feature.path.step)
        print(f"\n🔹 [{i+1}] {fname}")

        if feature.HasField('num_stats'):
            n = feature.num_stats
            print(f"   Type    : NUMERICAL")
            print(f"   Min     : {n.min:.4f}")
            print(f"   Max     : {n.max:.4f}")
            print(f"   Mean    : {n.mean:.4f}")
            print(f"   Std Dev : {n.std_dev:.4f}")
            print(f"   Missing : {n.num_zeros}")

        elif feature.HasField('string_stats'):
            s = feature.string_stats
            top = [b.value for b in s.top_values[:5]]
            print(f"   Type    : CATEGORICAL")
            print(f"   Unique  : {s.unique}")
            print(f"   Top 5   : {top}")
            print(f"   Avg Len : {s.avg_length:.2f}")

        else:
            print(f"   Type    : OTHER / BYTES")
            print(f"   (no numeric or string stats available)")

print("\n" + "=" * 60)
print(f"✅ Total features displayed: {len(dataset.features)}")

Number of datasets: 1

📊 Dataset: '' | Rows: 1381

🔹 [1] Age
   Type    : NUMERICAL
   Min     : 14.0000
   Max     : 61.0000
   Mean    : 37.3461
   Std Dev : 13.8890
   Missing : 0

🔹 [2] CAEC
   Type    : CATEGORICAL
   Unique  : 4
   Top 5   : ['Sometimes', 'Frequently', 'Always', 'no']
   Avg Len : 6.92

🔹 [3] CALC
   Type    : CATEGORICAL
   Unique  : 4
   Top 5   : ['Frequently', 'no', 'Always', 'Sometimes']
   Avg Len : 6.71

🔹 [4] CH2O
   Type    : NUMERICAL
   Min     : 1.0000
   Max     : 3.0000
   Mean    : 1.9942
   Std Dev : 0.7017
   Missing : 0

🔹 [5] FAF
   Type    : NUMERICAL
   Min     : 0.0000
   Max     : 3.0000
   Mean    : 1.5127
   Std Dev : 0.9510
   Missing : 215

🔹 [6] FAVC
   Type    : CATEGORICAL
   Unique  : 2
   Top 5   : ['yes', 'no']
   Avg Len : 2.52

🔹 [7] FCVC
   Type    : NUMERICAL
   Min     : 1.0000
   Max     : 3.0000
   Mean    : 2.0145
   Std Dev : 0.6891
   Missing : 0

🔹 [8] Gender
   Type    : CATEGORICAL
   Unique  : 2
   Top 5   : ['Male',

In [4]:
import tensorflow_data_validation as tfdv
from tensorflow_metadata.proto.v0 import statistics_pb2, anomalies_pb2

# ── Step 3: Read Schema ──────────────────────────────────────────
schema = tfdv.load_schema_text(
    '/root/airflow/dags/COMP315_Group8_project/outputs/SchemaGen/schema/3/schema.pbtxt'
)
print("=" * 60)
print("📋 SCHEMA — Feature Types & Constraints")
print("=" * 60)
for feature in schema.feature:
    print(f"\n🔹 {feature.name}")
    print(f"   Type      : {feature.type}")
    if feature.HasField('int_domain'):
        d = feature.int_domain
        print(f"   Domain    : INT  min={d.min}  max={d.max}")
    elif feature.HasField('float_domain'):
        d = feature.float_domain
        print(f"   Domain    : FLOAT  min={d.min}  max={d.max}")
    elif feature.HasField('string_domain'):
        d = feature.string_domain
        print(f"   Domain    : CATEGORICAL  values={list(d.value)}")
    print(f"   Presence  : min_fraction={feature.presence.min_fraction}")

# ── Step 4: Read Anomalies ───────────────────────────────────────
print("\n\n" + "=" * 60)
print("🔍 EXAMPLE VALIDATOR — Anomaly Report")
print("=" * 60)

for split in ['Split-train', 'Split-eval']:
    path = f'/root/airflow/dags/COMP315_Group8_project/outputs/ExampleValidator/anomalies/4/{split}/SchemaDiff.pb'
    with open(path, 'rb') as f:
        anomalies = anomalies_pb2.Anomalies()
        anomalies.ParseFromString(f.read())

    print(f"\n📂 {split}")
    print("-" * 40)
    if not anomalies.anomaly_info:
        print("   ✅ No anomalies found — this split is HEALTHY")
    else:
        for feature, info in anomalies.anomaly_info.items():
            print(f"   ❌ Feature  : {feature}")
            print(f"      Issue   : {info.description}")
            print(f"      Severity: {info.severity}")

# ── Pipeline Health Summary ──────────────────────────────────────
print("\n\n" + "=" * 60)
print("🏥 PIPELINE HEALTH CHECK")
print("=" * 60)
train_ok = not bool(anomalies.anomaly_info)
print(f"   ExampleGen   : ✅ Data ingested and split successfully")
print(f"   StatisticsGen: ✅ Statistics computed for all features")
print(f"   SchemaGen    : ✅ Schema inferred from training data")
print(f"   Validator    : {'✅ No anomalies detected' if train_ok else '⚠️  Anomalies detected — review above'}")
print(f"\n   Overall Status: {'✅ PIPELINE IS HEALTHY' if train_ok else '⚠️  PIPELINE NEEDS ATTENTION'}")

/root/anaconda3/envs/tfx-env/lib/python3.7/site-packages/google/auth/crypt/_cryptography_rsa.py:22: CryptographyDeprecationWarning: Python 3.7 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.7.
  import cryptography.exceptions
2026-04-06 15:38:10.776954: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-06 15:38:11.136488: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-06 15

📋 SCHEMA — Feature Types & Constraints

🔹 Age
   Type      : 2
   Presence  : min_fraction=1.0

🔹 CAEC
   Type      : 1
   Presence  : min_fraction=1.0

🔹 CALC
   Type      : 1
   Presence  : min_fraction=1.0

🔹 CH2O
   Type      : 3
   Presence  : min_fraction=1.0

🔹 FAF
   Type      : 3
   Presence  : min_fraction=1.0

🔹 FAVC
   Type      : 1
   Presence  : min_fraction=1.0

🔹 FCVC
   Type      : 3
   Presence  : min_fraction=1.0

🔹 Gender
   Type      : 1
   Presence  : min_fraction=1.0

🔹 Height
   Type      : 3
   Presence  : min_fraction=1.0

🔹 MTRANS
   Type      : 1
   Presence  : min_fraction=1.0

🔹 NCP
   Type      : 3
   Presence  : min_fraction=1.0

🔹 NObeyesdad
   Type      : 1
   Presence  : min_fraction=1.0

🔹 Obese
   Type      : 2
   Presence  : min_fraction=1.0

🔹 SCC
   Type      : 1
   Presence  : min_fraction=1.0

🔹 SMOKE
   Type      : 1
   Presence  : min_fraction=1.0

🔹 TUE
   Type      : 3
   Presence  : min_fraction=1.0

🔹 Weight
   Type      : 3
   Presence  